In [17]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.load_data import load_raw
df = load_raw()
print(df.shape)

(101766, 50)


c:\Users\bvksr\readmission-prediction\src\load_data.py:12: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path, na_values="?")


In [2]:
counts = df.patient_nbr.value_counts()
print("rows:", len(df))
print("unique patients:", df.patient_nbr.nunique())
print("max encounters for one patient:", counts.max())
print("median:", counts.median())
print("share of patients with >1 encounter:", (counts > 1).mean().round(4))
print("share of ROWS that are repeat encounters:", (len(df) - df.patient_nbr.nunique()) / len(df))

rows: 101766
unique patients: 71518
max encounters for one patient: 40
median: 1.0
share of patients with >1 encounter: 0.2345
share of ROWS that are repeat encounters: 0.29723090226598275


In [3]:
df.readmitted.value_counts(dropna=False)

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

In [4]:
first = df.sort_values("encounter_id").drop_duplicates("patient_nbr", keep="first")

In [5]:
print("full:", (df.readmitted == "<30").mean())
print("first-encounter:", (first.readmitted == "<30").mean())
print("first-encounter rows:", len(first))

full: 0.11159915885462728
first-encounter: 0.08799183422355211
first-encounter rows: 71518


In [6]:
nulls = first.isna().mean().sort_values(ascending=False)
nulls[nulls > 0]

weight               0.960108
max_glu_serum        0.951677
A1Cresult            0.818423
medical_specialty    0.482074
payer_code           0.434059
race                 0.027238
diag_3               0.017129
diag_2               0.004111
diag_1               0.000154
dtype: float64

In [7]:
first.nunique().sort_values().head(15)

citoglipton                 1
glimepiride-pioglitazone    1
examide                     1
acetohexamide               2
tolbutamide                 2
troglitazone                2
metformin-rosiglitazone     2
glipizide-metformin         2
tolazamide                  2
diabetesMed                 2
change                      2
metformin-pioglitazone      2
acarbose                    3
gender                      3
max_glu_serum               3
dtype: int64

In [12]:
print("unique patients == rows:", filtered.patient_nbr.nunique() == len(filtered))
print("death/hospice remaining:", filtered.discharge_disposition_id.isin(src.clean.DEATH_HOSPICE_IDS).sum())
print("positive rate:", (filtered.readmitted == "<30").mean())

unique patients == rows: True
death/hospice remaining: 0
positive rate: 0.08970602946850928


In [13]:
from src.clean import _drop_uninformative

reduced = _drop_uninformative(filtered)
print(reduced.shape)

dropping 14 columns:
  weight: explicit
  chlorpropamide: near-constant (0.9990)
  acetohexamide: near-constant (1.0000)
  tolbutamide: near-constant (0.9998)
  acarbose: near-constant (0.9971)
  miglitol: near-constant (0.9997)
  troglitazone: near-constant (1.0000)
  tolazamide: near-constant (0.9996)
  examide: constant
  citoglipton: constant
  glipizide-metformin: near-constant (0.9999)
  glimepiride-pioglitazone: constant
  metformin-rosiglitazone: near-constant (1.0000)
  metformin-pioglitazone: near-constant (1.0000)
(69973, 36)


In [14]:
from src.clean import _encode_missing, _build_target

encoded = _encode_missing(reduced)
final = _build_target(encoded)
print(final.shape)
print(final[["A1Cresult", "max_glu_serum", "race"]].apply(lambda s: s.value_counts()).T)

missing-as-category: ['A1Cresult', 'max_glu_serum'] -> 'NotTested'
                     ['race', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3'] -> 'Missing'
target 'readmitted_30d': 6277 positives, rate 0.0897
(69973, 36)
                >200   >300      >7      >8  AfricanAmerican  Asian  \
A1Cresult        NaN    NaN  2865.0  6239.0              NaN    NaN   
max_glu_serum  936.0  712.0     NaN     NaN              NaN    NaN   
race             NaN    NaN     NaN     NaN          12625.0  488.0   

               Caucasian  Hispanic  Missing    Norm  NotTested   Other  
A1Cresult            NaN       NaN      NaN  3741.0    57128.0     NaN  
max_glu_serum        NaN       NaN      NaN  1700.0    66625.0     NaN  
race             52292.0    1500.0   1918.0     NaN        NaN  1150.0  


In [16]:
from src.load_data import load_raw
from src.clean import clean

df_clean = clean(load_raw())
print(df_clean.shape)
df_clean.head()

rows: 101766 -> 71518 (first encounter) -> 69973 (alive)
dropping 14 columns:
  weight: explicit
  chlorpropamide: near-constant (0.9990)
  acetohexamide: near-constant (1.0000)
  tolbutamide: near-constant (0.9998)
  acarbose: near-constant (0.9971)
  miglitol: near-constant (0.9997)
  troglitazone: near-constant (1.0000)
  tolazamide: near-constant (0.9996)
  examide: constant
  citoglipton: constant
  glipizide-metformin: near-constant (0.9999)
  glimepiride-pioglitazone: constant
  metformin-rosiglitazone: near-constant (1.0000)
  metformin-pioglitazone: near-constant (1.0000)
missing-as-category: ['A1Cresult', 'max_glu_serum'] -> 'NotTested'
                     ['race', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3'] -> 'Missing'
target 'readmitted_30d': 6277 positives, rate 0.0897
(69973, 36)


,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,...,glimepiride,glipizide,glyburide,pioglitazone,rosiglitazone,insulin,glyburide-metformin,change,diabetesMed,readmitted_30d
0,12522,48330783,Caucasian,Female,[80-90),2,1,4,13,Missing,...,No,Steady,No,No,No,Steady,No,Ch,Yes,0
1,15738,63555939,Caucasian,Female,[90-100),3,3,4,12,Missing,...,No,No,No,No,Steady,Steady,No,Ch,Yes,0
2,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,Missing,...,No,Steady,No,No,No,Steady,No,Ch,Yes,0
3,28236,89869032,AfricanAmerican,Female,[40-50),1,1,7,9,Missing,...,No,No,No,No,No,Steady,No,No,Yes,0
4,35754,82637451,Caucasian,Male,[50-60),2,1,2,3,Missing,...,No,No,No,No,No,Steady,No,No,Yes,0


In [18]:
from src.load_data import load_raw
from src.clean import clean

df_clean = clean(load_raw(), verbose=False)

print(df_clean.diag_1.nunique(), df_clean.diag_2.nunique(), df_clean.diag_3.nunique())
print(df_clean.diag_1.value_counts().head(15))
print([c for c in df_clean.diag_1.unique() if not str(c).replace(".", "").isdigit()][:20])

695 724 757
diag_1
414      5209
428      3876
786      3040
410      2774
486      2362
427      2019
715      1907
434      1514
682      1463
780      1409
491      1313
276      1180
996      1106
250.8    1074
38        998
Name: count, dtype: int64
['V57', 'V58', 'V55', 'V53', 'Missing', 'V45', 'V26', 'V71', 'V56', 'V67', 'V60', 'V54', 'V43', 'V63', 'V25', 'V70', 'E909', 'V51']


In [19]:
from src.icd9 import add_diagnosis_groups

df_grouped = add_diagnosis_groups(df_clean)
print(df_grouped.shape)
print(df_grouped.diag_1_group.value_counts())

(69973, 36)
diag_1_group
Circulatory        21384
Other              12122
Respiratory         9486
Digestive           6487
Diabetes            5748
Injury              4694
Musculoskeletal     4064
Genitourinary       3440
Neoplasms           2538
Missing               10
Name: count, dtype: int64


In [20]:
mask = df_grouped.diag_1_group == "Other"
print(df_clean.loc[mask, "diag_1"].value_counts().head(20))

diag_1
682    1463
780    1409
276    1180
38      998
V57     654
296     634
789     365
278     357
8       275
295     275
648     242
285     211
280     207
707     156
331     133
386     116
V58     115
784     108
348      87
654      81
Name: count, dtype: int64
